# 04 — Multivariate anomaly detection and diagnosis: PCA vs MSCVAE

Two detectors behind the same `Detector` interface, evaluated by the same code against known events
(curated blackouts + ONS load interruptions ≥ 1 GW):

* **PCA** — Hotelling T² (variation inside the model) and SPE/Q (variation the model cannot explain),
  limits = 99% quantile of a KDE fit on the reference; diagnosis by reconstruction-based
  contributions (RBC).
* **MSCVAE** — the author's multi-scale convolutional VAE on signature matrices (`anomaly/mscvae.py`,
  through `anomaly/mscvae_adapter.py`), SPOT (extreme-value) threshold calibrated on held-out normal
  data, diagnosis by its root-cause analysis (RCA).

**Data.** Hourly load, hydro, thermal, wind, solar and net interchange per subsystem + SE nuclear,
as *weekly-baseline residuals* (x_t minus the mean of the same hour in the previous 4 weeks), which
removes the daily/weekly cycles. Sources below 1% of the subsystem load in the reference are dropped.

**Protocol.** Fit on two normal windows (Jun/2022–Apr/2023 and Sep/2023–Aug/2024, known events
removed), score 2010 → last day. An alarm needs 3 of the last 4 hours above the limit. FAR = share of
normal hours in alarm (outside the reference, ≥ 2 days from any known event).

> MSCVAE uses torch (no LightGBM in this notebook). `RUN_MSCVAE = False` skips it (~30–60 min on an M4).

In [ ]:
%matplotlib inline
import pandas as pd

from ons_energy.anomaly.experiment import MSPCConfig, matched_operating_point, run_mspc
from ons_energy.tracking import log_mspc
from ons_energy.viz import mspc_plots as P

pd.set_option("display.precision", 3)
LOG_MLFLOW = True
RUN_MSCVAE = True

## Experiments

In [ ]:
CONFIGS = [MSPCConfig(name="pca", detector="pca", params={"var_explained": 0.9, "alpha": 0.01, "name": "PCA"})]
if RUN_MSCVAE:
    CONFIGS.append(MSPCConfig(name="mscvae", detector="mscvae", params={
        "window_sizes": [6, 24, 168], "seq_len": 6, "stride": 2, "epochs": 15, "batch_size": 128,
        "val_fraction": 0.2, "q": 0.001, "name": "MSCVAE"}))

results = {}
for cfg in CONFIGS:
    results[cfg.name] = run_mspc(cfg)
    if LOG_MLFLOW:
        log_mspc(cfg, results[cfg.name])
pd.DataFrame([r["summary"] for r in results.values()])

## Comparison at the same false-alarm rate
Each detector's own limit sits at a different operating point, so the threshold on statistic/limit
is moved until the FAR on normal hours is 5%, and detection is compared there.

In [ ]:
pd.DataFrame([matched_operating_point(r, target_far=0.05) for r in results.values()])

## SIN blackout of 15/08/2023 (shaded)
Statistic / control limit (log scale; limit = 1); dots are persistent alarms.

In [ ]:
S = {name: r["scores"] for name, r in results.items()}
panels = [(S["pca"], "T2", "PCA T²"), (S["pca"], "SPE", "PCA SPE")]
if RUN_MSCVAE:
    panels.append((S["mscvae"], "phi", "MSCVAE φ"))
fig = P.control_chart(panels, "2023-08-12", "2023-08-19", ("2023-08-15 08:30", "2023-08-15 15:00"), "Fast anomaly: SIN blackout of 15/08/2023")

## N/NE blackout of 21/03/2018 (shaded)

In [ ]:
fig = P.control_chart(panels, "2018-03-18", "2018-03-25", ("2018-03-21 15:48", "2018-03-21 21:00"), "Fast anomaly: N/NE blackout of 21/03/2018")

## Diagnosis: which variables explain each alarm?
PCA: RBC to T² and SPE (each normalized by its limit). MSCVAE: the author's RCA. Averaged over the
alarmed hours of the event.

In [ ]:
items = [(results["pca"]["contributions"], "PCA — RBC")]
if RUN_MSCVAE:
    items.append((results["mscvae"]["contributions"], "MSCVAE — RCA"))
for ev in ("blackout_2023_08_15", "blackout_2018_03_21"):
    fig = P.contributions(items, ev, f"Who explains the {ev[9:].replace('_', '/')} alarm?")

## Events
Curated blackouts, then the most recent large load interruptions.

In [ ]:
COLS = ["event_id", "subsystems", "detected", "delay_h", "max_ratio", "alarm_share", "top_contributors"]
ev = pd.concat({name: r["events"] for name, r in results.items()}, names=["detector"]).reset_index(level=0)
ev[ev.kind == "blackout"][["detector", *COLS]]

In [ ]:
ev[ev.kind == "interruption"].sort_values("start", ascending=False).groupby("detector").head(10)[["detector", *COLS]]

## Alarms over the years
Share of hours in alarm per year: the static reference ages as wind and solar grow.

In [ ]:
fig = P.far_by_year([(r["scores"], r["summary"]["detector"]) for r in results.values()], "Share of hours in alarm per year")